## Imports

In [1]:
from pathlib import Path
import glob
import re

import pandas as pd
import numpy as np

## Configurações

In [2]:
# Diretórios
RAW_DIR = Path("../../data/raw/GSE296007")
PROCESSED_DIR = Path("../../data/interim")

# Arquivos de saída
EXPRESSION_OUTPUT = PROCESSED_DIR / "microplastic_expression.csv"
METADATA_OUTPUT = PROCESSED_DIR / "microplastic_metadata.csv"

# Padrão de entrada
INPUT_PATTERN = str(RAW_DIR / "*.txt.gz")

# Garante que a pasta de saída exista
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Amostras esperadas no experimento
EXPECTED_SAMPLES = {
    "CTR_1", "CTR_2", "CTR_3",
    "MA1_1", "MA1_2", "MA1_3",
    "MB1_1", "MB1_2", "MB1_3",
    "MC1_1", "MC1_2", "MC1_3",
    "MD1_1", "MD1_2", "MD1_3",
    "MA100_1", "MA100_2", "MA100_3",
    "MB100_1", "MB100_2", "MB100_3",
    "MC100_1", "MC100_2", "MC100_3",
}

## Funções Auxiliares

In [3]:
def parse_sample_name(filename: str) -> str:
    """
    Extrai o nome lógico da amostra a partir do nome do arquivo.

    Exemplo:
        GSM8963286_CTR_1_count.txt.gz -> CTR_1
    """
    base = Path(filename).name
    parts = base.split("_")

    if len(parts) >= 3:
        return f"{parts[1]}_{parts[2]}"

    return Path(base).stem


def load_count_file(filepath: str) -> pd.Series:
    """
    Lê um arquivo .txt.gz de contagens por gene e retorna uma Series indexada por gene_id.
    """
    df = pd.read_csv(
        filepath,
        sep="\t",
        header=None,
        names=["gene_id", "count"],
        compression="gzip"
    )

    df = df[~df["gene_id"].astype(str).str.startswith("__")]
    return df.set_index("gene_id")["count"]


def parse_metadata_from_sample(sample_id: str) -> dict:
    """
    Constrói os metadados de uma amostra a partir do sample_id.
    
    Convenções adotadas no subconjunto de RNA-seq do estudo:
    - CTR   = controle
    - MA100 = 0.1 µm, 0.1 g/L
    - MB100 = 0.1 µm, 0.01 g/L
    - MC100 = 0.1 µm, 0.001 g/L
    - MA1   = 1 µm, 0.1 g/L
    - MB1   = 1 µm, 0.01 g/L
    - MC1   = 1 µm, 0.001 g/L
    - MD1   = 1 µm, 0.05 g/L

    O sufixo final (_1, _2, _3) representa a réplica biológica.
    """
    group_metadata_map = {
        "CTR": {
            "particle_type": "control",
            "particle_size_um": np.nan,
            "particle_size_nm": np.nan,
            "concentration_gL": 0.0,
            "is_control": True,
            "treatment_status": "control",
        },
        "MA100": {
            "particle_type": "polystyrene",
            "particle_size_um": 0.1,
            "particle_size_nm": 100,
            "concentration_gL": 0.1,
            "is_control": False,
            "treatment_status": "treated",
        },
        "MB100": {
            "particle_type": "polystyrene",
            "particle_size_um": 0.1,
            "particle_size_nm": 100,
            "concentration_gL": 0.01,
            "is_control": False,
            "treatment_status": "treated",
        },
        "MC100": {
            "particle_type": "polystyrene",
            "particle_size_um": 0.1,
            "particle_size_nm": 100,
            "concentration_gL": 0.001,
            "is_control": False,
            "treatment_status": "treated",
        },
        "MA1": {
            "particle_type": "polystyrene",
            "particle_size_um": 1.0,
            "particle_size_nm": 1000,
            "concentration_gL": 0.1,
            "is_control": False,
            "treatment_status": "treated",
        },
        "MB1": {
            "particle_type": "polystyrene",
            "particle_size_um": 1.0,
            "particle_size_nm": 1000,
            "concentration_gL": 0.01,
            "is_control": False,
            "treatment_status": "treated",
        },
        "MC1": {
            "particle_type": "polystyrene",
            "particle_size_um": 1.0,
            "particle_size_nm": 1000,
            "concentration_gL": 0.001,
            "is_control": False,
            "treatment_status": "treated",
        },
        "MD1": {
            "particle_type": "polystyrene",
            "particle_size_um": 1.0,
            "particle_size_nm": 1000,
            "concentration_gL": 0.05,
            "is_control": False,
            "treatment_status": "treated",
        },
    }

    match = re.match(r"^([A-Z0-9]+)_([123])$", sample_id)
    if not match:
        raise ValueError(f"Formato de sample_id não reconhecido: {sample_id}")

    group = match.group(1)
    replicate = int(match.group(2))

    if group not in group_metadata_map:
        raise ValueError(f"Grupo não reconhecido no mapeamento: {group}")

    metadata = group_metadata_map[group].copy()
    metadata.update({
        "sample_id": sample_id,
        "group": group,
        "replicate": replicate,
    })

    return metadata

## Carregamento das Amostras

In [4]:
# Carrega os arquivos de contagem e constrói a matriz de expressão
files = sorted(glob.glob(INPUT_PATTERN))

if not files:
    raise FileNotFoundError(f"Nenhum arquivo encontrado no padrão: {INPUT_PATTERN}")

sample_counts = {}

# Itera sobre os arquivos, extrai o nome da amostra e carrega as contagens
for filepath in files:
    sample_name = parse_sample_name(filepath)
    sample_counts[sample_name] = load_count_file(filepath)

# Constrói o DataFrame de expressão a partir do dicionário de contagens
expression_df = pd.DataFrame(sample_counts).reset_index()
expression_df = expression_df.rename(columns={"index": "gene_id"})

print(f"Arquivos encontrados: {len(files)}")
print(f"Dimensões da matriz: {expression_df.shape[0]} genes x {expression_df.shape[1] - 1} amostras")

expression_df.head()

Arquivos encontrados: 24
Dimensões da matriz: 63241 genes x 24 amostras


,gene_id,CTR_1,CTR_2,CTR_3,MA1_1,MA1_2,MA1_3,MD1_1,MD1_2,MD1_3,...,MC1_3,MA100_1,MA100_2,MA100_3,MB100_1,MB100_2,MB100_3,MC100_1,MC100_2,MC100_3
0,ENSG00000000003,357,327,329,276,226,355,276,365,336,...,370,336,285,323,333,233,334,310,358,398
1,ENSG00000000005,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ENSG00000000419,770,453,733,605,629,716,655,796,969,...,680,721,700,733,710,703,633,727,974,712
3,ENSG00000000457,27,115,31,26,33,35,42,38,47,...,27,30,31,7,22,40,44,41,68,67
4,ENSG00000000460,37,27,7,4,5,36,3,27,0,...,20,29,3,0,17,9,19,12,21,33


In [5]:
# Valida as amostras carregadas

loaded_samples = [col for col in expression_df.columns if col != "gene_id"]
loaded_set = set(loaded_samples)

missing_samples = EXPECTED_SAMPLES - loaded_set
unexpected_samples = loaded_set - EXPECTED_SAMPLES

print("Amostras carregadas:")
print(sorted(loaded_samples))
print(f"\nTotal de amostras carregadas: {len(loaded_samples)}")

if missing_samples:
    print("\nAmostras esperadas, mas ausentes:")
    print(sorted(missing_samples))
else:
    print("\nNenhuma amostra esperada está faltando.")

Amostras carregadas:
['CTR_1', 'CTR_2', 'CTR_3', 'MA100_1', 'MA100_2', 'MA100_3', 'MA1_1', 'MA1_2', 'MA1_3', 'MB100_1', 'MB100_2', 'MB100_3', 'MB1_1', 'MB1_2', 'MB1_3', 'MC100_1', 'MC100_2', 'MC100_3', 'MC1_1', 'MC1_2', 'MC1_3', 'MD1_1', 'MD1_2', 'MD1_3']

Total de amostras carregadas: 24

Nenhuma amostra esperada está faltando.


## Salvamento da Matriz de Expressão

In [6]:
# Salva a matriz de expressão

expression_df.to_csv(EXPRESSION_OUTPUT, index=False)
print(f"Matriz de expressão salva em: {EXPRESSION_OUTPUT}")

Matriz de expressão salva em: ../../data/interim/microplastic_expression.csv


## Criação da Tabela de Metadados

In [7]:
# Constrói a tabela de metadados a partir dos sample_ids
metadata_records = [parse_metadata_from_sample(sample_id) for sample_id in loaded_samples]
metadata_df = pd.DataFrame(metadata_records)

# Ordena a tabela de metadados para facilitar a visualização
metadata_df = metadata_df.sort_values(by=["is_control", "group", "replicate"], ascending=[False, True, True]).reset_index(drop=True)

print(f"Dimensões da tabela de metadados: {metadata_df.shape[0]} amostras x {metadata_df.shape[1]} colunas")
metadata_df

Dimensões da tabela de metadados: 24 amostras x 9 colunas


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,sample_id,group,replicate
0,control,NaN,NaN,0.000,True,control,CTR_1,CTR,1
1,control,NaN,NaN,0.000,True,control,CTR_2,CTR,2
2,control,NaN,NaN,0.000,True,control,CTR_3,CTR,3
3,polystyrene,1.0,1000.0,0.100,False,treated,MA1_1,MA1,1
4,polystyrene,1.0,1000.0,0.100,False,treated,MA1_2,MA1,2
5,polystyrene,1.0,1000.0,0.100,False,treated,MA1_3,MA1,3
6,polystyrene,0.1,100.0,0.100,False,treated,MA100_1,MA100,1
7,polystyrene,0.1,100.0,0.100,False,treated,MA100_2,MA100,2
8,polystyrene,0.1,100.0,0.100,False,treated,MA100_3,MA100,3
9,polystyrene,1.0,1000.0,0.010,False,treated,MB1_1,MB1,1


In [8]:
# Salva a tabela de metadados

metadata_df.to_csv(METADATA_OUTPUT, index=False)

print(f"Tabela de metadados salva em: {METADATA_OUTPUT}")

Tabela de metadados salva em: ../../data/interim/microplastic_metadata.csv
